In [1]:
import os
import sys
from matplotlib.colors import ListedColormap
import numpy as np
import matplotlib.pyplot as plt

cm = ListedColormap(np.fromfile("/home/zhanh0f/Downloads/colorbar_ind.r@", dtype=np.float32).reshape(3,256).T)

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader

sys.path.append("../")

from package.datasets import Dataset
from package.networks.unet import UNetModel


def denormalize(x_norm, min_val=1000, max_val=5000):
        return (x_norm + 1) / 2 * (max_val - min_val) + min_val
        
def plot_formal(tensor, savepath, savename, dx=10, dz=10):

    # ---- Denormalize (user-defined) ----
    tensor = denormalize(tensor)
    data = tensor.detach().cpu().numpy().squeeze()


    # convert extent to km
    dh = 12.5*2  # grid spacing (meters)

    # convert extent to km
    extent = [
        275  * dh / 1000,
        (275+ data.shape[1]) * dh / 1000,
        data.shape[0] * dh* 0.5 / 1000,
        0 ]

    vmin, vmax = 1000, 4800

    # -------- TRUE MODEL --------
    fig, ax = plt.subplots(figsize=(10/1.5,5/1.5))

    im = ax.imshow(data,  cmap=cm, vmin=vmin, vmax=vmax,
                aspect='auto', extent=extent)

    cbar = plt.colorbar(im)
    cbar.set_label('Velocity (m/s)')

    ax.set_xlabel('Distance (km)', fontsize=12)
    ax.set_ylabel('Depth (km)', fontsize=12)

    # move distance axis to top
    ax.xaxis.set_label_position('top')
    ax.xaxis.tick_top()
    ax.set_yticks([0, 1, 2, 3, 4])
    ax.set_yticklabels(['0.0', '1.0', '2.0', '3.0 ', '4.0'])
    plt.tight_layout()
    
    plt.savefig(f"{savepath}/{savename}.png", dpi=600, bbox_inches="tight")
    plt.close(fig)

    

## Model Initialization and Configuration

This cell initializes the pretrained diffusion model used for velocity model generation and prepares the inference environment for subsequent sampling experiments.  
The pretrained network parameters are loaded from a saved checkpoint, and the model is transferred to the GPU device for efficient computation.

The model operates in inference mode only, with all parameters frozen to prevent weight updates during the generation process.  
Conditional information is incorporated through class-label embeddings integrated into the time-embedding pathway, enabling conditional generation without using classifier-free guidance (CFG).

In addition, this cell defines the output directory structure used to store generated results and ensures that the overall experimental configuration is properly initialized before sampling begins.

In [2]:


batch_size = 1

use_cfg = False
device = 'cuda'
checkpoint_path = './checkpoints/unet_110.pth' 


option = 'post_fwi_viking'
save_path = '../results/' +option 
index_f = 'FWI_Well'
def print_config(**kwargs):
    width = max(len(k) for k in kwargs) + 2
    print("\n" + "=" * 50)
    print(" Model Configuration ".center(50, "="))
    print("=" * 50)
    for k, v in kwargs.items():
        print(f"{k:<{width}}: {v}")
    print("=" * 50 + "\n")


print_config(

    batch_size=batch_size,
    save_path=save_path,
    use_cfg=use_cfg,
    device=device,
    checkpoint_path=checkpoint_path
)
os.makedirs(save_path, exist_ok=True)

model = UNetModel(
    image_size=320,
    in_channels=1,
    out_channels=1,
    num_classes=2,
    model_channels=192,
    channel_mult=(1,2,4,8),
    num_res_blocks=2,

    attention_resolutions=[80,40],
    num_head_channels=64,

    dropout=0.05,
    use_scale_shift_norm=True,
    resblock_updown=True,
)

model.to(device)
model.eval()




checkpoint = torch.load(checkpoint_path)
model.load_state_dict(checkpoint['model'])
for param in model.parameters():
        param.requires_grad = False


============== Model Configuration ===============
batch_size       : 1
save_path        : ../results/post_fwi_viking
use_cfg          : False
device           : cuda
checkpoint_path  : ./checkpoints/unet_110.pth



## Dataset Preparation

This cell prepares the datasets and dataloaders used during the sampling and evaluation stages of the experiment.  
The annotation files corresponding to the FWI results and reference velocity models are first specified and converted into complete file paths.

Two datasets are then constructed: one containing the input FWI velocity models and the other containing the corresponding ground-truth models.  
Both datasets are preloaded into memory to improve data access efficiency during inference.

Finally, PyTorch dataloaders are initialized for sequential batch processing without random shuffling, ensuring that the generated results remain aligned with their corresponding reference models during evaluation and visualization.

In [3]:
anno_path='../package/split_files'

test_anno = 'Viking.txt'





test_anno=os.path.join(anno_path, test_anno)



test_dataset=Dataset(
        test_anno,
        preload=True,
        lines=1,
        file_size=batch_size,
    )

test_dataloader = DataLoader(test_dataset, num_workers=1, batch_size=batch_size, shuffle=False)   



../data/viking_data/ifwi_viking_ep400.npy


#prepared the two well logs needed for well log guidance

In [4]:
wells = torch.from_numpy(
    np.load('/home/zhanh0f/Downloads/well_vp_interp.npy')[:, ::2]
).float().to(device)

wells = 2 * (wells - 1000) / (5000-1000) - 1

## Posterior Sampling with Observation-Guided Flow Matching

This cell implements the posterior sampling procedure used for conditional velocity model generation.  
The sampling process combines the learned prior information from the pretrained Flow Matching model with external observational constraints derived from Full Waveform Inversion (FWI) results and well-log information.

A Gaussian smoothing operator is first constructed and applied to the generated velocity model during sampling.  
The purpose of this smoothing operation is to extract the low-frequency structural components of the generated model, which are then compared with the observed FWI result.  
This allows the optimization process to preserve the large-scale structures recovered by FWI while still maintaining the high-frequency prior information learned by the generative model.

In addition, a total variation (TV) regularization term is introduced to stablize the optimization.

The generation process follows a reverse ODE trajectory starting from an intermediate noisy state.  
At each sampling step, the pretrained Flow Matching model predicts the transport velocity field, which is used to evolve the latent variable toward the data distribution.

To incorporate observational constraints, gradient-based guidance is applied during each reverse ODE step using three loss terms:

- An FWI consistency loss computed between the smoothed generated model and the observed FWI result.
- A well-log consistency loss enforcing agreement at selected spatial locations.
- A TV regularization loss.

The gradients of the combined objective are normalized and used to guide the latent variable toward solutions that simultaneously satisfy the observational constraints and remain consistent with the learned prior distribution of realistic velocity models.

This procedure effectively performs posterior sampling under physical and geological constraints while preserving the generative capabilities of the pretrained Flow Matching model.

In [5]:
def gaussian_kernel2d(sigma, device):
    """
    Create a normalized 2D Gaussian kernel.
    """
    radius = int(3 * sigma)
    ksize = 2 * radius + 1

    ax = torch.arange(-radius, radius + 1, device=device)

    xx, yy = torch.meshgrid(ax, ax, indexing='ij')

    kernel = torch.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    kernel = kernel / kernel.sum()

    return kernel.view(1, 1, ksize, ksize)


def gaussian_smooth_2d(x, sigma=3):
    """
    Apply Gaussian smoothing with reflection padding.
    """

    kernel = gaussian_kernel2d(sigma, x.device)

    pad = kernel.shape[-1] // 2

    x_pad = F.pad(
        x,
        (pad, pad, pad, pad),
        mode='reflect'
    )

    return F.conv2d(x_pad, kernel)


def tv_loss(x):
    """
    Total variation regularization.
    """

    dx = x[..., 1:] - x[..., :-1]
    dy = x[:, :, 1:, :] - x[:, :, :-1, :]

    return dx.abs().mean() + dy.abs().mean()


def bandpass_filter_torch(f_low, f_high, x, dt, pad=(125,125), order=8):
    """
    Differentiable Butterworth band-pass filter

    f_low  : high-pass cutoff
    f_high : low-pass cutoff
    x      : (..., nt)
    """

    original_shape = x.shape

    # make 3D for reflect padding
    x = x.reshape(1,1,-1)

    if pad is not None:
        x = F.pad(x, pad, mode="reflect")

    nt = x.shape[-1]
    device = x.device

    freqs = torch.fft.fftfreq(nt, d=dt).to(device).abs()
    eps = 1e-12

    # high-pass
    H_hp = 1.0 / torch.sqrt(
        1.0 + (f_low / (freqs + eps)) ** (2 * order)
    )

    # low-pass
    H_lp = 1.0 / torch.sqrt(
        1.0 + (freqs / (f_high + eps)) ** (2 * order)
    )

    # band-pass
    H = H_hp * H_lp

    X = torch.fft.fft(x, dim=-1)
    Xf = X * H
    x = torch.fft.ifft(Xf, dim=-1).real

    if pad is not None:
        left, right = pad
        x = x[..., left:-right]

    return x.reshape(original_shape)
# ======================================
# Sampling Configuration
# ======================================

ODE_step = 100
start_time = 30

dt = 1.0 / ODE_step

type = 1  # 0: Otway prior, 1: CGG prior


# ======================================
# Sampling Loop
# ======================================
y=torch.tensor([type,]).long().to(device) 

for iter_idx, test_data in enumerate(test_dataloader):

    batch_bar = tqdm(
        range(start_time, ODE_step - 6),
        desc="Generation steps",
        leave=False,
    )

    x_1 = test_data[0].to(device)



    # Observation used for guidance
    x_obs = x_1

    seeds = range(2, 3)

    posterior = []

    for seed in seeds:

        # ----------------------------------
        # Random Initialization
        # ----------------------------------

        g = torch.Generator(device=device).manual_seed(seed)


        noise = torch.randn(
            x_1.shape,
            generator=g,
            device=device,
        )

        x_t = (
            (start_time / ODE_step) * x_1
            + (1 - (start_time / ODE_step)) * noise
        )

        mask = torch.ones_like(x_t)

        latent = []

        loss_latent = []

        # ----------------------------------
        # Reverse ODE Sampling
        # ----------------------------------

        for index, j in enumerate(batch_bar):

            j = int(j)

            ratio = (ODE_step - j) / ODE_step

            lr = 2.5e0 * ratio ** 1.25

            x_t = x_t.detach().requires_grad_(True)

            t = torch.tensor(
                [j * dt],
                device=device,
            )

            # ----------------------------------
            # Flow Matching Prediction
            # ----------------------------------

  


            v_pred = model(
                x=x_t,
                timesteps=t,
                y=y,
            )

            x11 = x_t + (1 - t) * v_pred

            if j % 10 == 0:
                latent.append(
                    x11.detach().clone()
                )

            x11 = x11.clamp(-1, 1)

            # ----------------------------------
            # Observation Guidance
            # ----------------------------------

            x_sys1=gaussian_smooth_2d(x11[...,:110,:],sigma=0.1)
            x_sys2=gaussian_smooth_2d(x11[...,110:,:],sigma=3)    

            x_sys = torch.cat([x_sys1,x_sys2],dim=2) 

            loss_fwi = (
                F.mse_loss(
                    x_sys,
                    x_obs,
                    reduction='none',
                ) * mask
            ).mean()

            loss_well = (
                F.mse_loss(
                    bandpass_filter_torch(0.025,500,x11[...,100:,[int(807/2-275),int(1571/2-275)]],1),
                    bandpass_filter_torch(0.025,500,wells[100:,[int(807/2),int(1571/2)]],1) ,
                    reduction='none',
                ).mean()
       
            )

            loss_tv = tv_loss(x11)

            loss = (
                1.0 * loss_fwi
                + 0.5 * loss_well
                + 0.005 * loss_tv
            )

            # ----------------------------------
            # Gradient Guidance
            # ----------------------------------

            grad = torch.autograd.grad(
                loss,
                x_t,
            )[0]

            grad = grad / (
                grad.norm() + 1e-8
            )

            with torch.no_grad():

                # Guidance update
                x_t -= lr * grad

            # ----------------------------------
            # ODE Transport Step
            # ----------------------------------

            x_t = x_t + v_pred * dt

            loss_latent.append(
                loss.item()
            )

            batch_bar.set_postfix({
                'Loss_fwi': f'{loss_fwi.item():.6f}',
                'Loss_well': f'{loss_well.item():.6f}',
            })

        posterior.append(x11)

    generation_tensor = torch.stack(
        latent,
        dim=0,
    )

posterior_tensor = torch.stack(
    posterior,
    dim=0,
)

(1, 1, 320, 608)
(640, 2281) None


Generation steps:   0%|          | 0/64 [00:00<?, ?it/s]/tmp/ipykernel_2073772/1414641980.py:213: UserWarning: Using a target size (torch.Size([220, 2])) that is different to the input size (torch.Size([1, 1, 220, 2])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  F.mse_loss(


In [ ]:
# =====================================
# Save Visualization Results
# =====================================

plot_formal(
    x11,
    save_path,
    f'processed_fwi_{index_f}.png'
)

plot_formal(
    x_1,
    save_path,
    f'original_fwi_{index_f}.png'
)


plot_formal(
    x_sys,
    save_path,
    f'smoothed_observation_{index_f}.png'
)

